In [ ]:
import os
import zipfile
import shutil
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets,transforms,models
from torch.utils.data import DataLoader

ZIP_NAME='Dataset.zip'
EXTRACT_PATH='/content/sat_data'

if os.path.exists(EXTRACT_PATH):
	shutil.rmtree(EXTRACT_PATH)

if os.path.exists(ZIP_NAME):
	with zipfile.ZipFile(ZIP_NAME,'r') as zip_ref:
		zip_ref.extractall(EXTRACT_PATH)
	print("Extraction Complete.")
else:
	raise FileNotFoundError(f"Missing {ZIP_NAME}! Please upload it to the Colab sidebar.")

def find_true_root(path):
	for root,dirs,files in os.walk(path):
		if len(dirs)==10:
			return root
	return None

DATA_DIR=find_true_root(EXTRACT_PATH)

if not DATA_DIR:
	DATA_DIR=EXTRACT_PATH

print(f"Verified Data Root: {DATA_DIR}")
print(f"Classes Found: {os.listdir(DATA_DIR)}")

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE=32
EPOCHS=30

transform=transforms.Compose([
	transforms.Resize((224,224)),
	transforms.RandomHorizontalFlip(),
	transforms.RandomVerticalFlip(),
	transforms.RandomRotation(90),
	transforms.ToTensor(),
	transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

dataset=datasets.ImageFolder(root=DATA_DIR,transform=transform)
NUM_CLASSES=len(dataset.classes)

targets=torch.tensor(dataset.targets)
class_counts=torch.bincount(targets)
weights=1.0/class_counts.float()
class_weights=(weights/weights.sum()*NUM_CLASSES).to(DEVICE)

train_loader=DataLoader(dataset,batch_size=BATCH_SIZE,shuffle=True)

print(f"Training on {len(dataset)} images across {NUM_CLASSES} classes.")

class SatelliteAutoencoder(nn.Module):
	def __init__(self):
		super().__init__()
		self.encoder=nn.Sequential(
			nn.Conv2d(3,32,3,padding=1),
			nn.ReLU(),
			nn.MaxPool2d(2,2),
			nn.Conv2d(32,64,3,padding=1),
			nn.ReLU(),
			nn.MaxPool2d(2,2)
		)
		self.decoder=nn.Sequential(
			nn.ConvTranspose2d(64,32,3,stride=2,padding=1,output_padding=1),
			nn.ReLU(),
			nn.ConvTranspose2d(32,16,3,stride=2,padding=1,output_padding=1),
			nn.ReLU(),
			nn.Conv2d(16,3,3,padding=1),
			nn.Sigmoid()
		)

	def forward(self,x):
		return self.decoder(self.encoder(x))

classifier=models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)

classifier.avgpool=nn.AdaptiveAvgPool2d((1,1))

classifier.classifier=nn.Sequential(
	nn.Flatten(),
	nn.Linear(512,256),
	nn.ReLU(True),
	nn.Dropout(0.5),
	nn.Linear(256,NUM_CLASSES)
)

classifier.to(DEVICE)

autoencoder=SatelliteAutoencoder().to(DEVICE)

criterion_cls=nn.CrossEntropyLoss(weight=class_weights)
criterion_rec=nn.MSELoss()

optimizer=optim.Adam(list(classifier.parameters())+list(autoencoder.parameters()),lr=1e-4)

print(f"Starting Training on {DEVICE}...")

for epoch in range(EPOCHS):
	classifier.train()
	autoencoder.train()
	running_loss=0.0

	for images,labels in train_loader:
		images,labels=images.to(DEVICE),labels.to(DEVICE)

		optimizer.zero_grad()

		reconstructed=autoencoder(images)

		outputs=classifier(reconstructed)

		loss_rec=criterion_rec(reconstructed,images)
		loss_cls=criterion_cls(outputs,labels)

		total_loss=loss_cls+(loss_rec*0.5)

		total_loss.backward()

		optimizer.step()

		running_loss+=total_loss.item()

	avg_loss=running_loss/len(train_loader)

	print(f"Epoch [{epoch+1}/{EPOCHS}] - Combined Loss: {avg_loss:.4f}")

torch.save(classifier.state_dict(),'vgg16_model.pth')

torch.save(autoencoder.state_dict(),'autoencoder_model.pth')

print("Complete! Download the .pth files from the Colab file sidebar.")